In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os, joblib
import numpy as np
import pandas as pd
import polars as pl

import kaggle_evaluation.default_inference_server

from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from sklearn.linear_model import RidgeCV

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.model_selection import train_test_split

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split

# Read data
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

def advanced_feature_engineering(data):
    """Create advanced features from raw data"""
    df = data.copy()
    
    # 1. Economic Features (E-series)
    e_features = [f'E{i}' for i in range(1, 21)]
    
    # Statistical aggregations
    df['E_mean'] = df[e_features].mean(axis=1)
    df['E_std'] = df[e_features].std(axis=1)
    df['E_median'] = df[e_features].median(axis=1)
    df['E_max'] = df[e_features].max(axis=1)
    df['E_min'] = df[e_features].min(axis=1)
    df['E_range'] = df['E_max'] - df['E_min']
    df['E_skew'] = df[e_features].skew(axis=1)
    df['E_kurt'] = df[e_features].kurtosis(axis=1)
    
    # Volatility measures
    df['E_coeff_var'] = df['E_std'] / (df['E_mean'].abs() + 1e-10)
    
    # Percentile features
    df['E_q25'] = df[e_features].quantile(0.25, axis=1)
    df['E_q75'] = df[e_features].quantile(0.75, axis=1)
    df['E_iqr'] = df['E_q75'] - df['E_q25']
    
    # 2. Interaction features between key variables
    if 'S1' in df.columns and 'S2' in df.columns:
        df['S1_S2_ratio'] = df['S1'] / (df['S2'].abs() + 1e-10)
        df['S1_S2_product'] = df['S1'] * df['S2']
        df['S1_S2_diff'] = df['S1'] - df['S2']
    
    if 'S5' in df.columns and 'S1' in df.columns:
        df['S5_S1_ratio'] = df['S5'] / (df['S1'].abs() + 1e-10)
    
    if 'P8' in df.columns and 'P9' in df.columns:
        df['P8_P9_ratio'] = df['P8'] / (df['P9'].abs() + 1e-10)
        df['P8_P9_product'] = df['P8'] * df['P9']
    
    if 'P10' in df.columns and 'P12' in df.columns:
        df['P10_P12_ratio'] = df['P10'] / (df['P12'].abs() + 1e-10)
    
    # 3. Polynomial features for key indicators
    key_features = ['S1', 'S2', 'S5', 'P8', 'P9', 'I2']
    for feat in key_features:
        if feat in df.columns:
            df[f'{feat}_squared'] = df[feat] ** 2
            df[f'{feat}_cubed'] = df[feat] ** 3
            df[f'{feat}_sqrt'] = np.sign(df[feat]) * np.sqrt(np.abs(df[feat]))
    
    # 4. Rolling statistics (if temporal ordering exists)
    # Note: This assumes data has temporal structure
    window_sizes = [3, 5, 10]
    for feat in ['E_mean', 'S1', 'S2', 'P9']:
        if feat in df.columns:
            for window in window_sizes:
                df[f'{feat}_roll_mean_{window}'] = df[feat].rolling(window=window, min_periods=1).mean()
                df[f'{feat}_roll_std_{window}'] = df[feat].rolling(window=window, min_periods=1).std()
    
    # 5. Momentum features
    for feat in ['E_mean', 'S1', 'S2', 'P9']:
        if feat in df.columns:
            df[f'{feat}_momentum_3'] = df[feat].diff(3)
            df[f'{feat}_momentum_5'] = df[feat].diff(5)
    
    # 6. Binning and categorical encodings
    if 'E_mean' in df.columns:
        df['E_mean_bin'] = pd.cut(df['E_mean'], bins=5, labels=False)
    
    # 7. Distance from mean
    for feat in ['S1', 'S2', 'P9', 'P8']:
        if feat in df.columns:
            df[f'{feat}_dist_from_mean'] = df[feat] - df[feat].mean()
            df[f'{feat}_z_score'] = (df[feat] - df[feat].mean()) / (df[feat].std() + 1e-10)
    
    return df

def preprocessing(data, typ, scaler=None):
    """Enhanced preprocessing with feature engineering"""
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19',
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',
                     "S2", "P9", "S1", "S5", "I2", "P8",
                     "P10", "P12", "P13"]
    
    # Create feature-engineered dataset
    data_enhanced = advanced_feature_engineering(data)
    
    if typ == "train":
        # Keep target
        if 'forward_returns' in data_enhanced.columns:
            data_enhanced = data_enhanced.rename(columns={'forward_returns': 'target'})
        
        # Select features
        feature_cols = [col for col in data_enhanced.columns if col != 'target']
        target_col = ['target']
        
        # Fill nulls in features
        for col in feature_cols:
            if data_enhanced[col].isnull().any():
                data_enhanced[col].fillna(data_enhanced[col].median(), inplace=True)
        
        # Drop rows with null targets
        data_enhanced = data_enhanced.dropna(subset=['target'])
        
        # Scale features
        if scaler is None:
            scaler = RobustScaler()  # More robust to outliers than StandardScaler
            data_enhanced[feature_cols] = scaler.fit_transform(data_enhanced[feature_cols])
        else:
            data_enhanced[feature_cols] = scaler.transform(data_enhanced[feature_cols])
        
        return data_enhanced, scaler
    
    else:  # inference
        feature_cols = [col for col in data_enhanced.columns]
        
        # Fill nulls
        for col in feature_cols:
            if data_enhanced[col].isnull().any():
                data_enhanced[col].fillna(data_enhanced[col].median(), inplace=True)
        
        # Scale features
        if scaler is not None:
            # Only transform columns that exist in the scaler
            common_cols = [col for col in feature_cols if col in scaler.feature_names_in_]
            data_enhanced[common_cols] = scaler.transform(data_enhanced[common_cols])
        
        return data_enhanced

# Apply preprocessing
train_processed, scaler = preprocessing(train, "train")

# Split data
train_split, val_split = train_test_split(
    train_processed, test_size=0.1, random_state=42, shuffle=False  # shuffle=False for time series
)

train_x = train_split.drop(columns=["target"])
test_x = val_split.drop(columns=["target"])
train_y = train_split['target']
test_y = val_split['target']

print(f"Original features: 29")
print(f"Enhanced features: {train_x.shape[1]}")
print(f"Training samples: {len(train_x)}")
print(f"Validation samples: {len(test_x)}")
print(f"\nSample of new features:")
print(train_x.columns.tolist()[:20])

Original features: 29
Enhanced features: 175
Training samples: 8091
Validation samples: 899

Sample of new features:
['date_id', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18']


In [3]:
train

,date_id,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,V3,V4,V5,V6,V7,V8,V9,forward_returns,risk_free_rate,market_forward_excess_returns
0,0,0,0,0,1,1,0,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.002421,0.000301,-0.003038
1,1,0,0,0,1,1,0,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.008495,0.000303,-0.009114
2,2,0,0,0,1,0,0,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.009624,0.000301,-0.010243
3,3,0,0,0,1,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.004662,0.000299,0.004046
4,4,0,0,0,1,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.011686,0.000299,-0.012301
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,8985,0,0,0,0,0,0,0,0,0,...,0.469577,0.837963,1.226772,0.822751,-0.707361,0.142857,-0.649616,0.002457,0.000155,0.001990
8986,8986,0,0,0,0,0,0,0,0,0,...,0.671958,0.837963,0.785877,0.805556,-0.715692,0.196098,-0.668289,0.002312,0.000156,0.001845
8987,8987,0,0,1,0,0,0,0,0,0,...,0.481481,0.787698,0.834898,0.823413,-0.723949,0.133929,-0.670946,0.002891,0.000156,0.002424
8988,8988,0,0,0,0,0,0,0,0,0,...,0.655423,0.783730,0.994026,0.851852,-0.684937,0.101852,-0.646265,0.008310,0.000156,0.007843


In [4]:
LGBM_R_parm = {'boosting_type': 'gbdt', 
               'colsample_bytree': 0.9484106149593443, 
               'learning_rate': 0.1988123373955639, 
               'max_bin': 77, 
               'max_depth': 10, 
               'metric': 'mape', 
               'min_child_samples': 81, 
               'min_data_in_leaf': 21, 
               'n_estimators': 5029, 
               'num_leaves': 42, 
               'objective': 'regression_l1', 
               'reg_alpha': 0.6355835028602363, 
               'reg_lambda': 3.109823217156622, 
               'subsample': 0.7300733288106989, 
               'verbosity': -1}

print(f'LGBMRegressor TRAINING...')
LGBM_R = LGBMRegressor(**LGBM_R_parm)
LGBM_R.fit(train_x, train_y, eval_set=[(test_x, test_y)])
joblib.dump(LGBM_R, 'LGBM_R.pkl')

LGBMRegressor TRAINING...


['LGBM_R.pkl']

In [5]:
LGBM_R_model = joblib.load('LGBM_R.pkl')

In [6]:
# Signal conversion parameters (like in working code)
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    """Convert raw prediction to signal in range [0.0, 2.0]"""
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)

def predict(test: pl.DataFrame) -> float:
    try:
        # Convert to pandas
        test_df = test.to_pandas()
        
        # Preprocess - use 'inference' type to handle test data properly
        test_processed = preprocessing(test_df, 'inference')
        #print(test_processed)
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        #print(float(signal))
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0  # Return neutral signal (midpoint) on error
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

Prediction error: Number of features of the model must match the input. Model n_features_ is 175 and input n_features is 177
Prediction error: Number of features of the model must match the input. Model n_features_ is 175 and input n_features is 177
Prediction error: Number of features of the model must match the input. Model n_features_ is 175 and input n_features is 177
Prediction error: Number of features of the model must match the input. Model n_features_ is 175 and input n_features is 177
Prediction error: Number of features of the model must match the input. Model n_features_ is 175 and input n_features is 177
Prediction error: Number of features of the model must match the input. Model n_features_ is 175 and input n_features is 177
Prediction error: Number of features of the model must match the input. Model n_features_ is 175 and input n_features is 177
Prediction error: Number of features of the model must match the input. Model n_features_ is 175 and input n_features is 177
